In [5]:
from src.config_manager import ConfigManager
from src.training_manager import TrainingManager
from src.experiment_manager import ExperimentManager
import torch
device = str(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print(device)

cpu


## Choose dataset and models

First, we decide on a name for the experiment we will be conducting (all results and models from this experiment will be saved in a folder with the given name).

Then, we decide on which dataset we will use for the experiment. Options are "lorenz", "spiral", "double_pendulum", "random_skew","logistic_map", "hopf" and "pod".

Then, we decide on the models to train and compare for this experiment. The options are "mlp", "transformer", "rnnautoreg","oldrnn", "esn", "nodemlp","koopmanmlp", "nodetransformer", "koopmantransformer","nodernnautoreg","koopmanrnnautoreg", "none" (check readme for more info). We can choose to train each model a number of times, to compare the effect of different initializations.

In [6]:

dataset_name = "lorenz"
experiment_name = f"{dataset_name}_experiment" 
models = [['none',1],['mlp',1],['transformer',1] ,['esn',1]]


## Train the models

In [7]:
config_manager = ConfigManager(dataset_name, device)
training_manager = TrainingManager(device)

data_handler_params = config_manager.get_current_dataset_config()

In [8]:
training_manager.train_multiple_models(experiment_name, config_manager, models)

CSV files have been saved in the 'results/lorenz_experiment/trajectories/train' directory.
CSV files have been saved in the 'results/lorenz_experiment/trajectories/val' directory.
CSV files have been saved in the 'results/lorenz_experiment/trajectories/test' directory.
No best‐val checkpoint found at results/lorenz_experiment/none1/best_current_model.pth. Using last‐epoch weights.
none1
Similarity matrices + metadata written to results/lorenz_experiment/mlp1
Similarity matrices + metadata written to results/lorenz_experiment/mlp1
Epoch 1/200  train=0.3373  val=0.4529  sim=0.7046  lr=9.50e-04  (0.3s)
Similarity matrices + metadata written to results/lorenz_experiment/mlp1
Epoch 2/200  train=0.0845  val=0.4002  sim=0.7434  lr=9.02e-04  (0.4s)
Similarity matrices + metadata written to results/lorenz_experiment/mlp1
Epoch 3/200  train=0.0372  val=0.3923  sim=0.7623  lr=8.57e-04  (0.4s)
Similarity matrices + metadata written to results/lorenz_experiment/mlp1
Epoch 4/200  train=0.0246  val=0

### Create relative latent spaces and similarity matrix

To compute the relative latent spaces, we need to choose a number of anchors and a number of samples to embed in the relative latent space.

In [9]:
anchors = 80
points_to_embed = 1000

experiment_manager = ExperimentManager(config_manager, device)

models, latent_spaces, absolute_latent_spaces = experiment_manager.evalute_latent_spaces_exp(experiment_name, points_to_embed, anchors, random_anchors=True)

Directory 'results/lorenz_experiment/trajectories/train' already contains files; skipping save.
Directory 'results/lorenz_experiment/trajectories/val' already contains files; skipping save.
Directory 'results/lorenz_experiment/trajectories/test' already contains files; skipping save.
Model loaded from results/lorenz_experiment/esn1/model.pkl
Torch‐loaded model from results/lorenz_experiment/mlp1/best_current_model.pth
Torch‐loaded model from results/lorenz_experiment/none1/model.pth
Torch‐loaded model from results/lorenz_experiment/transformer1/best_current_model.pth
Similarity matrices + metadata written to results/lorenz_experiment


### Benchmark the model performances
This will create a benchmark file in each models directory which will contain useful metrics that measure the models performance.

In [10]:
experiment_manager.benchmark_models_exp(experiment_name)

Model loaded from results/lorenz_experiment/esn1/model.pkl
Torch‐loaded model from results/lorenz_experiment/mlp1/best_current_model.pth
Torch‐loaded model from results/lorenz_experiment/none1/model.pth
Torch‐loaded model from results/lorenz_experiment/transformer1/best_current_model.pth
149450
149450
Benchmarks successfully saved to results/lorenz_experiment/esn1/benchmarks
esn1: 
 {'pdf_dim1 kl_divergence': 0.00033631066393105134, 'pdf_dim2 kl_divergence': 0.00023530707360461213, 'pdf_dim3 kl_divergence': 0.00031824846037498177, 'pdf_kl_divergence_avg': 0.0002966220659702151, 'mse': 0.0016817059971572905, 'rmse': 0.041008608817628654, 'mae': 0.010667877213156825}
152500
152500
Benchmarks successfully saved to results/lorenz_experiment/mlp1/benchmarks
mlp1: 
 {'pdf_dim1 kl_divergence': 0.01924104143951895, 'pdf_dim2 kl_divergence': 0.01998669042561113, 'pdf_dim3 kl_divergence': 0.04414559707414453, 'pdf_kl_divergence_avg': 0.02779110964642487, 'mse': 0.41144109987233923, 'rmse': 0.641

### Visualize relative latent spaces
We will create plots that visualize the relative latent spaces of different modes.

For this, we will choose a method to reduce the dimensionality (either "pca" or "umap"), and a list of the directories of the models we want to include in the plot.

The plots will be saved in the experiment directory.

In [ ]:
#choose which models latent spaces to visualize
#specify the individual plots like this: vis_latent = (reducer, [model1, model2]), for reducers [pca, umap] are available
vis_latent1 = ('pca',['mlp1','transformer1','esn1'])
vis_latent2 = ('umap', ['mlp1', 'esn1'])
vis_latent = [vis_latent1,vis_latent2]

experiment_manager.visualize_relative_latent_spaces(models, latent_spaces, vis_latent, experiment_name, fit_individually=False, n_components = 3)

/opt/anaconda3/envs/rg-py312/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### Visualize absolute latent spaces
It works the same as visualizing relative latent spaces but you need to pass the absolute_latent_spaces and
set fit_individually=True or fix_global_limits=True since the absolute latent spaces have different shapes.

In [ ]:
#choose which models latent spaces to visualize
#specify the individual plots like this: vis_latent = (reducetar, [model1, model2]), for reducers [pca, umap] are available

vis_latent1 = ('pca',['mlp1','transformer1','esn1'])
vis_latent2 = ('umap', ['mlp1', 'esn1'])
vis_latent = [vis_latent1,vis_latent2]

                                                            #!                                                   #!                                       #!
experiment_manager.visualize_relative_latent_spaces(models, absolute_latent_spaces, vis_latent, experiment_name, fit_individually=True, n_components = 3, fix_global_limits = True)

/opt/anaconda3/envs/rg-py312/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
